# NC MNIST — Two-Phase Training (CE then MSE)

**Motivation:** MSE loss + wd=1e-3 prevents the model
from memorising training data (stuck at 97.9% — never hit 99% threshold).

**This fix:**
- **Phase 1 (200 epochs, CE loss):** reach 99%+ train acc easily
- **Phase 2 (300 epochs, MSE loss):** stay in terminal phase, drive NC1 down
- **wd=1e-4:** weak enough to memorise, strong enough for NC

This is what Papyan 2020 did in practice.

**Fully self-contained. Est: ~20 min on T4.**

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working/'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda  |  PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [2]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,),(0.3081,))])
trainset  = torchvision.datasets.MNIST('/kaggle/working/data',
    train=True,  download=True, transform=transform)
testset   = torchvision.datasets.MNIST('/kaggle/working/data',
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=256, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'MNIST: {len(trainset):,} train / {len(testset):,} test')


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 340kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]

MNIST: 60,000 train / 10,000 test


In [3]:
class MLP5(nn.Module):
    def __init__(self, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(4):
            layers += [nn.Linear(width, width), act_cls()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.head(self.body(x))
    def get_features(self, x):
        self(x); return self._feats
    def get_classifier_weights(self):
        return self.head.weight.detach()

# Quick sanity check
m = MLP5().to(DEVICE)
f = m.get_features(torch.randn(4,1,28,28).to(DEVICE))
print(f'Model OK — features: {tuple(f.shape)}  '
      f'params: {sum(p.numel() for p in m.parameters())/1e6:.2f}M')
del m, f


Model OK — features: (4, 512)  params: 1.46M


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask]-(-1.0/(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(DEVICE),y.to(DEVICE)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total

print('Metrics ready.')


Metrics ready.


In [5]:
def train_nc_twophase(model, name='m', lr=1e-3, wd=1e-4,
                      phase1_epochs=200, phase2_epochs=300,
                      nc_every=10):
    """
    Phase 1 (CE loss, 200 ep): reach 99%+ train accuracy quickly.
    Phase 2 (MSE loss, 300 ep): stay in terminal phase, drive NC1 collapse.
    wd=1e-4: weak enough to allow memorisation, strong enough for NC.
    """
    model = model.to(DEVICE)
    K     = 10
    rows, terminal, t0 = [], False, time.time()

    for phase, loss_type, n_ep in [
        (1, 'ce',  phase1_epochs),
        (2, 'mse', phase2_epochs),
    ]:
        opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        start_ep = (phase1_epochs if phase==2 else 0)
        print(f'  Phase {phase} ({loss_type.upper()} loss, {n_ep} epochs)')

        for ep_local in range(1, n_ep+1):
            ep = start_ep + ep_local
            model.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_type == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y,K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sched.step()

            if ep_local % nc_every == 0 or ep_local == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'    [{name}] Terminal phase at epoch {ep}')
                nc = compute_nc(model, train_loader) if terminal else \
                     {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'    ep={ep:>4} tr={tr:.4f} te={te:.4f} '
                      f'nc1={nc1s} fn={fns} t={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < 0.01:
                    print(f'    *** NC1 < 0.01 — full collapse at epoch {ep}!')
                    return pd.DataFrame(rows)
    return pd.DataFrame(rows)

print('train_nc_twophase ready.')


train_nc_twophase ready.


In [6]:
# Baseline: MLP-5, ReLU, wd=1e-4, Phase1=CE 200ep, Phase2=MSE 300ep
# Expected: terminal phase by ep ~50-80, NC1<0.01 by ep ~300-400
# Expected time: ~20 min on T4
print('=== Two-phase baseline: CE->MSE | ReLU | wd=1e-4 ===')
torch.manual_seed(0)
model_base = MLP5(width=512, act_cls=nn.ReLU)
df_base = train_nc_twophase(
    model_base, name='MLP5-ReLU-twophase',
    lr=1e-3, wd=1e-4,
    phase1_epochs=200, phase2_epochs=300,
    nc_every=10)
df_base.to_csv(SAVE_DIR + 'mnist_twophase.csv', index=False)

nc_done = df_base.dropna(subset=['nc1'])
if len(nc_done):
    print(f'\nFinal NC1:       {nc_done.nc1.iloc[-1]:.6f}')
    print(f'Final NC2:       {nc_done.nc2.iloc[-1]:.6f}')
    print(f'Final NC3:       {nc_done.nc3.iloc[-1]:.6f}')
    print(f'Final feat_norm: {nc_done.feat_norm.iloc[-1]:.4f}')
    print(f'Final test acc:  {df_base.test.iloc[-1]:.4f}')
    t_rows = nc_done[nc_done.nc1 < 0.01]
    if len(t_rows):
        t_nc = t_rows.epoch.iloc[0]
        fn   = t_rows.feat_norm.iloc[0]
        print(f'\nT_NC (NC1<0.01): epoch {t_nc}  feat_norm={fn:.4f}')
        print('SUCCESS — full NC collapse achieved!')
    else:
        print(f'\nNC1 lowest so far: {nc_done.nc1.min():.6f}')
        print('Still collapsing — extend phase2_epochs if needed.')
else:
    print('Never reached terminal phase — check output above.')
print('Saved: mnist_twophase.csv')


=== Two-phase baseline: CE->MSE | ReLU | wd=1e-4 ===
  Phase 1 (CE loss, 200 epochs)
    [MLP5-ReLU-twophase] Terminal phase at epoch 10
    ep=  10 tr=0.9938 te=0.9786 nc1=0.21514 fn=26.776 t=1.4m
    ep=  20 tr=0.9966 te=0.9823 nc1=0.15798 fn=24.599 t=2.8m
    ep=  30 tr=0.9975 te=0.9835 nc1=0.13747 fn=22.369 t=4.1m
    ep=  40 tr=0.9961 te=0.9812 nc1=0.12635 fn=20.813 t=5.5m
    ep=  50 tr=0.9989 te=0.9840 nc1=0.10880 fn=19.785 t=6.9m
    ep=  60 tr=0.9988 te=0.9833 nc1=0.11203 fn=18.802 t=8.2m
    ep=  70 tr=0.9987 te=0.9837 nc1=0.10685 fn=18.090 t=9.6m
    ep=  80 tr=0.9996 te=0.9844 nc1=0.09561 fn=17.235 t=10.9m
    ep=  90 tr=1.0000 te=0.9859 nc1=0.08725 fn=16.583 t=12.3m
    ep= 100 tr=0.9997 te=0.9846 nc1=0.09416 fn=15.530 t=13.6m
    ep= 110 tr=0.9999 te=0.9856 nc1=0.08672 fn=15.321 t=15.0m
    ep= 120 tr=1.0000 te=0.9864 nc1=0.08136 fn=15.248 t=16.3m
    ep= 130 tr=0.9991 te=0.9839 nc1=0.09017 fn=13.005 t=17.7m
    ep= 140 tr=1.0000 te=0.9858 nc1=0.08896 fn=14.443 t=19.1m
  

In [7]:
if len(df_base.dropna(subset=['nc1'])) == 0:
    print('No NC data yet — run the training cell first.')
else:
    plt.rcParams.update({'font.family':'serif','font.size':11,
        'axes.spines.top':False,'axes.spines.right':False})
    nc_df = df_base.dropna(subset=['nc1'])
    t_nc  = nc_df[nc_df.nc1<0.01].epoch.iloc[0] \
            if (nc_df.nc1<0.01).any() else None
    # Phase boundary
    p2_start = df_base[df_base.phase==2].epoch.iloc[0] \
               if 2 in df_base.phase.values else None

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    for ax in axes:
        if p2_start:
            ax.axvline(p2_start, color='blue', ls=':', lw=1.2,
                       alpha=0.5, label='CE->MSE')

    axes[0].plot(df_base.epoch, df_base.train,
                 color='#2196F3', lw=2, label='Train')
    axes[0].plot(df_base.epoch, df_base.test,
                 color='#F44336', lw=2, ls='--', label='Test')
    axes[0].axhline(0.99, color='gray', ls=':', lw=1)
    axes[0].set(xlabel='Epoch', ylabel='Accuracy', title='(a) Accuracy')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.25)

    axes[1].semilogy(nc_df.epoch, nc_df.nc1, color='#4CAF50', lw=2)
    axes[1].axhline(0.01, color='black', ls=':', lw=1.2, label='NC1=0.01')
    if t_nc:
        axes[1].axvline(t_nc, color='black', ls='--', lw=1.2,
                        label=f'T_NC={t_nc}')
    axes[1].set(xlabel='Epoch', ylabel='NC1 (log)', title='(b) NC1 collapse')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.25)

    axes[2].plot(nc_df.epoch, nc_df.nc2,
                 color='#FF9800', lw=2, label='NC2 (ETF)')
    axes[2].plot(nc_df.epoch, nc_df.nc3,
                 color='#9C27B0', lw=2, ls='--', label='NC3 (self-dual)')
    axes[2].set(xlabel='Epoch', ylabel='Deviation',
                title='(c) NC2 + NC3')
    axes[2].legend(fontsize=9); axes[2].grid(alpha=0.25)

    axes[3].plot(nc_df.epoch, nc_df.feat_norm, color='#E91E63', lw=2)
    if t_nc:
        fn_val = nc_df[nc_df.epoch==t_nc].feat_norm.iloc[0]
        axes[3].axvline(t_nc, color='black', ls='--', lw=1.2,
                        label=f'T_NC  fn={fn_val:.3f}')
        axes[3].axhline(fn_val, color='#E91E63', ls=':', lw=1)
        axes[3].legend(fontsize=9)
    axes[3].set(xlabel='Epoch', ylabel='Feature norm',
                title='(d) Feature norm')
    axes[3].grid(alpha=0.25)

    fig.suptitle('Two-phase: CE->MSE | MLP-5 | ReLU | wd=1e-4 | MNIST',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(SAVE_DIR + 'fig_twophase.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig_twophase.png')


Saved: fig_twophase.png
